# 워밍업 — 같은 모델, 다른 대답 🦁🐕

웹 챕터에서 본 실험을 이번엔 **직접 코드로** 만져 봅니다.

> **노트북 사용법**: 각 셀(회색 상자)을 클릭하고 `Shift + Enter`를 누르세요. 위에서 아래로 반복하면 끝.
> 코드를 다 이해하지 못해도 괜찮아요. 실행해서 결과를 눈으로 보는 게 목적입니다.

**이 노트북의 목표**: 6부 전체가 풀 문제를 한 문장으로 얻는다.
**"조련 전 모델은 비서가 아니라 이어쓰기 기계다."**

In [ ]:
# 🤖 AI 튜터 — 공부하다 막히면 오른쪽 아래 버튼을 눌러 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('sample')

## 1. 실험: 같은 프롬프트, 두 모델

베이스 모델(조련 전)과 챗 모델(조련 후)의 **실제 행동을 재현한 대표 응답**을 나란히 비교합니다.
아래 셀을 실행하고, 베이스 모델이 "틀린" 게 아니라 **다른 일을 하고 있다**는 걸 관찰하세요.

In [ ]:
# 같은 프롬프트에 대한 두 모델의 전형적 응답 (베이스 모델의 실제 행동을 재현한 예시)
EXAMPLES = [
    {
        'prompt': '2 더하기 2는',
        'base': '2 더하기 2는 4라는 것을 모르는 사람은 없다. 하지만 우리나라 수학 교육의 현실을 들여다보면…',
        'chat': '4입니다.',
    },
    {
        'prompt': '아래 문장을 영어로 번역해 줘: 오늘 날씨가 참 좋다',
        'base': '아래 문장을 일본어로 번역해 줘: 내일 비가 온다고 한다\n아래 문장을 중국어로 번역해 줘: …',
        'chat': 'The weather is really nice today.',
    },
    {
        'prompt': '서울에서 꼭 가봐야 할 곳은 어디야?',
        'base': '라고 물으신다면 저는 할 말이 많죠ㅋㅋ 지난 주말에 다녀온 후기 올립니다~ 1편은 성수동…',
        'chat': '경복궁, 북촌 한옥마을, N서울타워를 추천해요.',
    },
]

for ex in EXAMPLES:
    print('프롬프트 :', ex['prompt'])
    print('  🦁 베이스(조련 전):', ex['base'])
    print('  🐕 챗   (조련 후):', ex['chat'])
    print()

## 2. 초소형 '이어쓰기 기계'를 직접 만들기

베이스 모델의 본업(다음 단어 예측)을 아주 작게 재현해 봅니다.
몇 개의 문장을 읽혀서 **"이 단어 다음엔 어떤 단어가 자주 오나"** 표를 만들고,
가장 자주 오는 단어를 계속 이어 붙이면, 그게 이어쓰기 기계입니다.
(진짜 LLM은 이걸 수만 배 크게, 확률로 부드럽게 하는 것. 4부에서 조립했던 그 원리)

In [ ]:
from collections import Counter, defaultdict

# 이 기계가 '인터넷'이라고 알고 있는 전부 — 블로그체 문장 몇 개
CORPUS = [
    '오늘 날씨가 참 좋다 그래서 공원에 나가 산책을 했다',
    '오늘 날씨가 참 좋다 이런 날은 커피 한 잔이 딱이다',
    '오늘 점심은 뭘 먹을까 고민하다가 김밥을 먹었다',
    '주말에 서울 나들이를 다녀온 후기 올립니다 경복궁은 역시 멋졌다',
]

# '단어 → 다음에 온 단어들의 횟수' 표 만들기 (이게 학습의 전부!)
next_words = defaultdict(Counter)
for line in CORPUS:
    words = line.split()
    for a, b in zip(words, words[1:]):
        next_words[a][b] += 1

def continue_text(prompt, length=8):
    """프롬프트 마지막 단어에서 출발해, 가장 자주 본 다음 단어를 계속 이어 붙인다."""
    words = prompt.split()
    for _ in range(length):
        last = words[-1]
        if last not in next_words:      # 이 단어 뒤를 본 적이 없으면 멈춤
            break
        words.append(next_words[last].most_common(1)[0][0])
    return ' '.join(words)

print(continue_text('오늘 날씨가'))
print(continue_text('주말에'))

그럴듯하게 이어 쓰죠? 그럼 이번엔 이 기계에게 **질문**을 해 봅시다.
비서라면 "김밥이요"라고 답하겠지만, 이어쓰기 기계는…

In [ ]:
# 질문을 던져도 — 답하는 게 아니라 그 문장 '다음에 올 법한 글'을 이어 쓸 뿐!
question = '오늘 점심은'
print('질문(이라고 생각하고 보낸 것):', question)
print('이어쓰기 기계의 반응        :', continue_text(question))

# 검증 — 기계는 프롬프트를 답변으로 바꾸지 않고, 뒤에 이어 붙이기만 한다
result = continue_text(question)
assert result.startswith(question), '이어쓰기 기계는 항상 원문 뒤에 이어 붙입니다!'
print('\n✔ 확인: 응답이 질문으로 시작한다 (답변이 아니라 이어쓰기라는 증거)')

## 3. 관찰 정리 + 🤖 AI 연결

- 방금 만든 초소형 기계도, 진짜 베이스 모델도 하는 일은 같습니다: **다음에 올 법한 단어를 이어 붙이기**.
- 질문에 답하지 못하는 건 능력 부족이 아니라 **행동 방식**의 문제였죠.
- 그래서 6부의 질문은 하나입니다: **"이어쓰기 기계를 어떻게 비서로 만드는가?"**
  - 1단계 사전학습(다음 단어 예측: 이미 완료, 4부) → 2단계 SFT(3장) → 3단계 선호 학습(RLHF·DPO, 4~7장)

**다음 챕터**: 1장(큰 그림). 이 3단 파이프라인의 전체 지도를 펼칩니다.

> 💡 궁금한 코드가 있었다면, 그 셀을 실행한 다음 셀에서 `from tutor import explain` 후 `explain('sample')`.
> 튜터가 코드를 한 줄씩 풀어 설명해 줍니다 (Ollama 실행 중일 때).